# CoRE Stack API Integration Demo

This notebook demonstrates the CoRE Stack APIs used by the Field Validator application.
All API calls are tested against the public CoRE Stack API endpoints.

**API Documentation**: [api-doc.core-stack.org](https://api-doc.core-stack.org/)

## Prerequisites

1. Obtain an API key from [core-stack.org/use-apis](https://core-stack.org/use-apis/)
2. Set the `CORESTACK_API_KEY` environment variable, or enter it when prompted below

In [1]:
import os
import requests
from pprint import pprint

# Configuration
BASE_URL = "https://api-doc.core-stack.org/api/v1"

# API key configuration
# Option 1: Set CORESTACK_API_KEY environment variable before running
# Option 2: Replace the empty string below with your API key
API_KEY = os.environ.get("CORESTACK_API_KEY", "")

if not API_KEY:
    # For interactive use - uncomment the line below:
    # API_KEY = input("Enter your CoRE Stack API key: ")
    print("⚠️ No API key configured. Set CORESTACK_API_KEY environment variable or edit this cell.")
else:
    print("✅ Configuration loaded")
    print(f"Base URL: {BASE_URL}")
    print(f"API Key: {'*' * 8}{API_KEY[-4:] if len(API_KEY) > 4 else '****'}")

# Common headers for all requests
HEADERS = {
    "Accept": "application/json",
    "X-API-Key": API_KEY
}

✅ Configuration loaded
Base URL: https://api-doc.core-stack.org/api/v1
API Key: ********hFgg


## Test Location

We'll use a sample location in the Western Ghats for testing.
This location is in Maharashtra near Panchgani.

In [2]:
# Test coordinates - Western Ghats location (near Panchgani, Maharashtra)
TEST_LAT = 17.9307
TEST_LON = 73.8023

print(f"Test Location: {TEST_LAT}, {TEST_LON}")
print(f"Google Maps: https://www.google.com/maps?q={TEST_LAT},{TEST_LON}")

Test Location: 17.9307, 73.8023
Google Maps: https://www.google.com/maps?q=17.9307,73.8023


---
## 1. Get Admin Details by Lat/Lon

**Endpoint**: `/get_admin_details_by_latlon/`

Returns administrative hierarchy (State, District, Tehsil) for given coordinates.

In [3]:
def get_admin_details(lat: float, lon: float) -> dict:
    """Get administrative details for a location."""
    url = f"{BASE_URL}/get_admin_details_by_latlon/"
    params = {"latitude": str(lat), "longitude": str(lon)}
    
    response = requests.get(url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()
    return response.json()

# Test the API
admin_details = get_admin_details(TEST_LAT, TEST_LON)
print("Admin Details Response:")
pprint(admin_details)

# Extract values for subsequent calls
STATE = admin_details.get("State", "")
DISTRICT = admin_details.get("District", "")
TEHSIL = admin_details.get("Tehsil", "")

print(f"\n📍 Location: {STATE} > {DISTRICT} > {TEHSIL}")

Admin Details Response:
{'District': 'SATARA', 'State': 'MAHARASHTRA', 'Tehsil': 'Mahabaleshwar'}

📍 Location: MAHARASHTRA > SATARA > Mahabaleshwar


---
## 2. Get MWS ID by Lat/Lon

**Endpoint**: `/get_mwsid_by_latlon/`

Returns the Micro-Watershed ID for a given location.

In [4]:
def get_mws_id(lat: float, lon: float) -> dict:
    """Get Micro-Watershed ID for a location."""
    url = f"{BASE_URL}/get_mwsid_by_latlon/"
    params = {"latitude": str(lat), "longitude": str(lon)}
    
    response = requests.get(url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()
    return response.json()

# Test the API
mws_result = get_mws_id(TEST_LAT, TEST_LON)
print("MWS ID Response:")
pprint(mws_result)

# Extract MWS ID for subsequent calls
MWS_ID = mws_result.get("uid", "")
print(f"\n🗺️ Micro-Watershed ID: {MWS_ID}")

MWS ID Response:
{'District': 'SATARA',
 'State': 'MAHARASHTRA',
 'Tehsil': 'Mahabaleshwar',
 'uid': '18_66653'}

🗺️ Micro-Watershed ID: 18_66653


---
## 3. Get Generated Layer URLs

**Endpoint**: `/get_generated_layer_urls/`

Returns available GIS layer URLs for a given tehsil. These are used to display vector/raster data on the map.

In [5]:
def get_layer_urls(state: str, district: str, tehsil: str) -> list:
    """Get generated layer URLs for a location."""
    url = f"{BASE_URL}/get_generated_layer_urls/"
    params = {"state": state, "district": district, "tehsil": tehsil}
    
    response = requests.get(url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()
    return response.json()

# Test the API
if STATE and DISTRICT and TEHSIL:
    layers = get_layer_urls(STATE, DISTRICT, TEHSIL)
    print(f"Found {len(layers)} layers:")
    for i, layer in enumerate(layers[:5]):  # Show first 5
        print(f"  {i+1}. {layer.get('layer_name', 'Unknown')} ({layer.get('layer_type', 'unknown')})")
    if len(layers) > 5:
        print(f"  ... and {len(layers) - 5} more")
else:
    print("⚠️ Skipping - admin details not available")

Found 83 layers:
  1. Terrain LULC (vector)
  2. Change Detection Vector (vector)
  3. Terrain LULC (vector)
  4. LULC_level_3 (raster)
  5. LULC_level_3 (raster)
  ... and 78 more


---
## 4. Get Tehsil Data

**Endpoint**: `/get_tehsil_data/`

Returns comprehensive data about a tehsil including area, population, and agricultural statistics.

In [6]:
def get_tehsil_data(state: str, district: str, tehsil: str) -> dict:
    """Get tehsil-level data."""
    url = f"{BASE_URL}/get_tehsil_data/"
    params = {"state": state, "district": district, "tehsil": tehsil}
    
    response = requests.get(url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()
    return response.json()

# Test the API
if STATE and DISTRICT and TEHSIL:
    tehsil_data = get_tehsil_data(STATE, DISTRICT, TEHSIL)
    print("Tehsil Data Response:")
    pprint(tehsil_data)
else:
    print("⚠️ Skipping - admin details not available")

Tehsil Data Response:
{'agroecological': [{'contact_person': 'Adinath Ombale',
                     'created_at': '2020-09-28 00:00:00+00',
                     'domains': '["Agroforestry","Cultivation '
                                'practices","Livelihoods","Soil conservation"]',
                     'email': 'shramik13@gmail.com',
                     'organization_name': 'Shramik Janata Vikas Sanstha',
                     'organization_type': 'Civil Society Organisation',
                     'uid': '18_65737'}],
 'aquifer_vector': [{'aquifer_class': 'Hard Rock',
                     'area_in_ha': 1936.41,
                     'principle_aq_alluvium_percent': 0,
                     'principle_aq_banded gneissic complex_percent': 0,
                     'principle_aq_basalt_percent': 100.0,
                     'principle_aq_charnockite_percent': 0,
                     'principle_aq_gneiss_percent': 0,
                     'principle_aq_granite_percent': 0,
                    

---
## 5. Get MWS KYL Indicators

**Endpoint**: `/get_mws_kyl_indicators/`

Returns "Know Your Landscape" indicators for a micro-watershed, including precipitation, runoff, cropping intensity, and more.

In [7]:
def get_kyl_indicators(state: str, district: str, tehsil: str, mws_id: str) -> list:
    """Get Know Your Landscape indicators for a micro-watershed."""
    url = f"{BASE_URL}/get_mws_kyl_indicators/"
    params = {
        "state": state,
        "district": district,
        "tehsil": tehsil,
        "mws_id": mws_id
    }
    
    response = requests.get(url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()
    return response.json()

# Test the API
if STATE and DISTRICT and TEHSIL and MWS_ID:
    kyl_data = get_kyl_indicators(STATE, DISTRICT, TEHSIL, MWS_ID)
    print("KYL Indicators Response:")
    pprint(kyl_data)
else:
    print("⚠️ Skipping - admin details or MWS ID not available")

KYL Indicators Response:
[{'aquifer_class': 0,
  'area_protection': 882.01,
  'area_wide_scale_restoration': 16.14,
  'avg_double_cropped': 32.5564,
  'avg_kharif_surface_water_mws': 92.4008,
  'avg_number_dry_spell': 1.125,
  'avg_precipitation': 846.6312,
  'avg_rabi_surface_water_mws': 92.1359,
  'avg_runoff': 216.1,
  'avg_single_cropped': 25.5752,
  'avg_triple_cropped': 7.3113,
  'avg_wsr_ratio_kharif': 13.1824,
  'avg_wsr_ratio_rabi': 19.7743,
  'avg_wsr_ratio_zaid': 94.1685,
  'avg_zaid_surface_water_mws': 76.2877,
  'built_up_area': 15.98,
  'cropping_intensity_avg': 1.1262,
  'cropping_intensity_trend': '0',
  'decrease_in_tree_cover': 40.28,
  'degradation_cropping_intensity': 61.7,
  'degradation_land_area': 45.98,
  'drought_category': 0,
  'factory_csr': 0,
  'green_credit': 0,
  'increase_in_tree_cover': 46.7,
  'lcw_conflict': 0,
  'lulc_plain_category': None,
  'lulc_slope_category': None,
  'mining': 0,
  'mws_id': '18_66653',
  'mws_intersect_villages': [562957,
    

---
## 6. Get Waterbodies by Admin Area

**Endpoint**: `/get_waterbodies_data_by_admin/`

Returns waterbody inventory for an administrative area.

In [9]:
def get_waterbodies(state: str, district: str, tehsil: str = None) -> dict:
    """Get waterbodies for an administrative area."""
    url = f"{BASE_URL}/get_waterbodies_data_by_admin/"
    params = {"state": state, "district": district}
    if tehsil:
        params["tehsil"] = tehsil
    
    response = requests.get(url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()
    return response.json()

# Test the API
if STATE and DISTRICT:
    waterbodies = get_waterbodies(STATE, DISTRICT, TEHSIL)
    if isinstance(waterbodies, list):
        print(f"Found {len(waterbodies)} waterbodies")
        if waterbodies:
            print("\nFirst waterbody:")
            pprint(waterbodies[0])
    else:
        # Response is a dictionary with count info
        print(f"Waterbodies Response:")
        pprint(waterbodies)
else:
    print("⚠️ Skipping - admin details not available")

Waterbodies Response:
{'18_55561_124': {'zoi_properties': {'UID': '18_55561_124',
                                     'cropping_intensity_2017': 0.9999999999999999,
                                     'cropping_intensity_2018': 1,
                                     'cropping_intensity_2019': 0.9571068108513531,
                                     'cropping_intensity_2020': 1,
                                     'cropping_intensity_2021': 1,
                                     'cropping_intensity_2022': 1,
                                     'cropping_intensity_2023': 0.9999999999999998,
                                     'doubly_cropped_area_2017': 0,
                                     'doubly_cropped_area_2018': 0,
                                     'doubly_cropped_area_2019': 0,
                                     'doubly_cropped_area_2020': 0,
                                     'doubly_cropped_area_2021': 0,
                                     'doubly_cropped_area_

---
## 7. Get MWS Time Series Data

**Endpoint**: `/get_mws_data/`

Returns time series data for a micro-watershed including ET, runoff, precipitation, and soil moisture.

In [11]:
def get_mws_data(mws_id: str, start_date: str = None, end_date: str = None) -> list:
    """Get MWS time series data."""
    url = f"{BASE_URL}/get_mws_data/"
    params = {"mws_id": mws_id}
    if start_date:
        params["start_date"] = start_date
    if end_date:
        params["end_date"] = end_date
    
    response = requests.get(url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()
    return response.json()

# Test the API
if MWS_ID:
    try:
        mws_data = get_mws_data(MWS_ID)
        print(f"Found {len(mws_data)} time series records")
        if mws_data:
            print("\nSample record:")
            pprint(mws_data[0])
    except requests.exceptions.HTTPError as e:
        print(f"⚠️ API returned error: {e}")
        print("Note: This endpoint may have server-side issues for certain MWS IDs")
else:
    print("⚠️ Skipping - MWS ID not available")

⚠️ API returned error: 500 Server Error: Internal Server Error for url: https://api-doc.core-stack.org/api/v1/get_mws_data/?mws_id=18_66653
Note: This endpoint may have server-side issues for certain MWS IDs


---
## 8. Get Active Locations

**Endpoint**: `/get_active_locations/`

Returns list of states with available data in CoRE Stack.

In [12]:
def get_active_locations() -> dict:
    """Get active locations with CoRE Stack coverage."""
    url = f"{BASE_URL}/get_active_locations/"
    
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    return response.json()

# Test the API
active_locations = get_active_locations()
print("Active Locations Response:")
pprint(active_locations)

Active Locations Response:
[{'district': [{'blocks': [{'block_id': '1566',
                            'label': 'Nallacheruvu',
                            'tehsil_id': '1566'},
                           {'block_id': '1914',
                            'label': 'Nambulipulikunta',
                            'tehsil_id': '1914'},
                           {'block_id': '1938',
                            'label': 'Raptadu',
                            'tehsil_id': '1938'},
                           {'block_id': '2033',
                            'label': 'Talupula',
                            'tehsil_id': '2033'}],
                'district_id': '185',
                'label': 'Ananthapur'},
               {'blocks': [{'block_id': '1600',
                            'label': 'Chowdepalle',
                            'tehsil_id': '1600'},
                           {'block_id': '2009',
                            'label': 'Ramakuppam',
                            'tehsil_id': '2009

---
## Summary

This notebook demonstrates all CoRE Stack APIs used by the Field Validator application:

| API Endpoint | Purpose | Status |
|-------------|---------|--------|
| `/get_admin_details_by_latlon/` | Get State/District/Tehsil for coordinates | ✅ |
| `/get_mwsid_by_latlon/` | Get Micro-Watershed ID | ✅ |
| `/get_generated_layer_urls/` | Get GIS layer URLs for map display | ✅ |
| `/get_tehsil_data/` | Get tehsil statistics | ✅ |
| `/get_mws_kyl_indicators/` | Get KYL indicators | ✅ |
| `/get_waterbodies_data_by_admin/` | Get waterbody inventory | ✅ |
| `/get_mws_data/` | Get time series data | ✅ |
| `/get_active_locations/` | Get available coverage areas | ✅ |

For more details, see the [CoRE Stack API documentation](https://api-doc.core-stack.org/).